# 25 · GSE135779 · scRNA_seq · WGCNA modules

Reads `wgcna_input.rds`. Writes `modules.rds` and the end product `module_genes.csv` in
`data/run_artifacts/GSE135779/`.

Settings as for GSE65391 (notebook 05): signed network and TOM, minModuleSize 30, mergeCutHeight 0.25.
There are more genes than fit one block, so `maxBlockSize = 5000`: WGCNA pre-clusters the genes into
blocks and builds modules within each block.

In [1]:
if (!requireNamespace("WGCNA", quietly = TRUE)) install.packages("WGCNA", repos = "https://cloud.r-project.org")
suppressMessages(library(WGCNA))
source("../src/paths.R")
w <- readRDS(art("GSE135779", "wgcna_input.rds"))
c(samples = nrow(w$datExpr), genes = ncol(w$datExpr), power = w$power)

samples   genes   power 
     33   15353      10

In [2]:
set.seed(SEED)
net <- blockwiseModules(w$datExpr, power = w$power, networkType = "signed", TOMType = "signed",
                        minModuleSize = 30, mergeCutHeight = 0.25, maxBlockSize = 5000,
                        numericLabels = FALSE, pamRespectsDendro = FALSE, verbose = 0)
sizes <- sort(table(net$colors), decreasing = TRUE)
c(modules = sum(names(sizes) != "grey"), genes_in_modules = sum(sizes[names(sizes) != "grey"]), grey = unname(sizes["grey"]))
sizes

modules genes_in_modules             grey 
              29            14147             1206


    turquoise          blue         brown          grey        yellow 
         2084          1870          1537          1206          1165 
        green           red         black          pink       magenta 
          771           733           511           439           400 
       purple   greenyellow           tan          cyan        salmon 
          360           349           342           335           335 
 midnightblue     lightcyan        grey60    lightgreen   lightyellow 
          322           320           307           303           262 
    royalblue       darkred     darkgreen darkturquoise      darkgrey 
          248           212           179           172           159 
       orange    darkorange         white       skyblue   saddlebrown 
          139            85            82            72            54 

**Result.** 29 modules holding 14,147 of 15,353 genes; 1,206 are grey.

In [3]:
ME  <- orderMEs(net$MEs)
kME <- cor(w$datExpr, ME)
modules <- setdiff(names(sizes), "grey")
hubs <- lapply(setNames(modules, modules), function(mo) {
  g <- names(net$colors)[net$colors == mo]
  names(sort(kME[g, paste0("ME", mo)], decreasing = TRUE))
})
t(sapply(hubs, function(h) paste(head(h, 8), collapse = ", ")))

turquoise,blue,brown,yellow,green,red,black,pink,magenta,purple,⋯,royalblue,darkred,darkgreen,darkturquoise,darkgrey,orange,darkorange,white,skyblue,saddlebrown
"FUS, HNRNPA1, RPSA, RPL13A, EIF3H, RPS3, RPL19, RPL13","EMILIN2, ATP11A, CTNNA1, CYBB, MFSD1, MPEG1, GNS, LPCAT2","EEIG1, DGKA, TOM1L2, CAMK4, DNMT3A, SHISAL2A, ABLIM1, EVL","SUN2, ARHGAP45, RASAL3, P2RY8, DVL2, AKAP8L, TRAPPC14, EML3","CCNB1, CDKN3, CCNA2, CDCA8, CD38, DCPS, CLSPN, DSCC1","GOLPH3L, LRIF1, JPT2, METTL13, CETN3, BOLA1, PAAF1, BRD8","BOLA2B, ATP5MGL, RP11-48B3.3, RP11-400N9.1, PCDH11Y, RP11-356C4.3, LINC01285, RP11-798L4.1","COX7A2, COX5B, PSMB1, PSMB7, ATP5PD, COXFA4, FUNDC2, CHCHD2","BLNK, CD19, FCRLA, CD79B, HLA-DOB, PNOC, CD72, MS4A1","MAP3K2, NUFIP2, GPCPD1, SAMD4B, CREBBP, JUND, PTBP3, HIPK1",⋯,"IFIT3, IFI35, OAS2, ISG15, RSAD2, OAS1, IFIT1, PARP9","CTD-2192J16.22, RP11-770J1.5, XXbac-BPG252P9.9, RP11-138I1.4, MTG1, MRPS24, FAM120B, NME2","CCL5, CST7, PPP2R2B, C1orf21, FCRL6, TGFBR3, GZMH, KIF21A","UNC93B1, ARHGAP27, EPS8, ABI3, C5orf15, SHISA4, PRKCD, PHETA1","NUDC, ADH5, CFAP298, THAP1, VRK3, TRUB2, HACD3, SDHAF2","CMTM5, TUBB1, ACRBP, SPARC, CTTN, CAVIN2, GP9, GFI1B","ZNF213-AS1, LINC01619, PRKN, CAPN7, LRP2BP, KPNB1-DT, FBXL14, RP11-159D12.2","PSMB3, POMP, NDUFS7, CYBA, RHOG, VPS29, SUPT4H1, JOSD2","ADRM1, PPP4C, UBE2A, TRIM26, LMNB1, ACTR1A, MTHFD2, CGAS","SLC4A1, SNCA, CA1, SELENBP1, LINC00570, TMC5-AS1, ALAS2, GYPB"


In [4]:
ifn6 <- read_gene_set("ifn-type1-6.txt")
data.frame(gene = ifn6, module = net$colors[ifn6], row.names = NULL)

gene,module
<chr>,<chr>
IFI27,royalblue
IFI44L,royalblue
IFIT1,royalblue
ISG15,royalblue
RSAD2,royalblue
SIGLEC1,royalblue


**Result.** All six interferon-score genes are in one module, royalblue. Other modules read by their
hub genes: green (CCNB1, CDKN3, CCNA2, CD38: cell cycle), orange (CMTM5, TUBB1, GP9: platelet),
saddlebrown (SLC4A1, CA1, GYPB, ALAS2: red cell), turquoise (RPSA, RPL13A, RPS3: ribosomal), magenta
(CD19, CD79B, MS4A1: B cell), darkgreen (CCL5, GZMH, FCRL6: NK / cytotoxic).

In [5]:
in_module <- names(net$colors)[net$colors != "grey"]
module_genes <- data.frame(gene = in_module, module = net$colors[in_module],
                           kME = kME[cbind(in_module, paste0("ME", net$colors[in_module]))], row.names = NULL)
write.csv(module_genes, art("GSE135779", "module_genes.csv"), row.names = FALSE)
saveRDS(list(colors = net$colors, MEs = ME, kME = kME, hubs = hubs), art("GSE135779", "modules.rds"))